# 01 — Getting started

**What you will learn**

- How to reach the GLiNER2 service and what to configure (`GLINER_BASE_URL`).
- The difference between the three GET routes — `/health`, `/health/deep` and
  `/version` — and which one belongs in a monitor, a container `HEALTHCHECK`,
  and a data-provenance record.
- Why the *architecture* the service reports decides which of the twelve routes
  you are allowed to call at all.
- Your first successful extraction, and how to read the response.

**What it assumes you already did**

Nothing. This is the first notebook in the path. You need Python 3 and
`requests`:

```bash
pip install requests
```

**Roughly how long**

About 10 minutes.

---

## What this service is

GLiNER2 is a *zero-shot information extraction* model. That phrase does a lot of
work, so it is worth unpacking before you send anything at it.

A conventional NER model is trained on a fixed tag set — `PERSON`, `ORG`, `LOC`
— and that tag set is baked into the weights. Wanting a new entity type means
labelling data and retraining. GLiNER2 inverts that: the *labels travel with the
request*. You send the text and the label set together, the model embeds your
label strings alongside the text, and it scores spans against them. The same
resident weights answer `["medication", "dosage", "symptom"]` on one request and
`["company", "person", "location"]` on the next, with no reload and no
fine-tuning between them.

That is the entire value proposition, and it shapes every design decision
downstream: schemas are cheap to change, but nothing is guaranteed to be stable
across model checkpoints, because you never trained anything.

The service wraps that model in twelve HTTP routes. Three are GETs for health
and identity; five do single-document inference; four do batched inference.
This notebook covers the three GETs and one inference call. The rest of the path
covers the other eight.

## Setup

`BASE_URL` is read from the `GLINER_BASE_URL` environment variable, with the
`jarvita-agx` LAN address as the fallback. Every notebook in this series uses
the same variable, so a single export redirects all seven of them.

| Deployment | Base URL |
|---|---|
| `jarvita-agx` on the LAN | `http://192.168.1.177:8013` |
| Container on the local host | `http://localhost:8013` |
| `make run` / `make dev` locally | `http://localhost:8125` |

The helpers below are re-declared at the top of every notebook so each one runs
standalone. Read them once here — you will not need to read them again.

The one genuinely non-obvious choice is `TIMEOUT = 130`. The server caps its own
inference at `REQUEST_TIMEOUT_SECONDS` (default 120) and answers `504` when it
exceeds that. Setting the client timeout *above* the server's means the server
always wins the race to give up, so a slow request comes back as a status code
you can log and act on rather than a bare client-side exception with no
information in it.

In [1]:
import json
import os
import time

import requests

# Every notebook in this path reads the same environment variable, so you can
# point the whole series at a different deployment with one export:
#     export GLINER_BASE_URL=http://localhost:8013
BASE_URL = os.environ.get("GLINER_BASE_URL", "http://192.168.1.177:8013")

# The server bounds its own inference at REQUEST_TIMEOUT_SECONDS (default 120)
# and returns 504 when it blows through that. A client timeout slightly above
# the server's means the server always gets to explain itself with a status
# code instead of the client giving up first and leaving you guessing.
TIMEOUT = 130

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})


def get(path):
    """GET a path and return parsed JSON. Raises on non-2xx."""
    r = session.get(f"{BASE_URL}{path}", timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post(path, payload):
    """POST JSON and return parsed JSON. Raises on non-2xx."""
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post_raw(path, payload):
    """POST JSON and return (status_code, parsed_body_or_text). Never raises.

    Used whenever the interesting part of the answer IS the status code.
    """
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text


def show(obj):
    """Pretty-print a JSON-serializable object."""
    print(json.dumps(obj, indent=2, ensure_ascii=False))


print("BASE_URL =", BASE_URL)

BASE_URL = http://192.168.1.177:8013


## The three GET routes answer three different questions

It is tempting to treat these as one thing with three spellings. They are not.

| Route | Runs inference? | What it actually proves | Where it belongs |
|---|---|---|---|
| `GET /health` | no | The HTTP process is alive and can describe itself | Container `HEALTHCHECK`, cheap load polling |
| `GET /health/deep` | **yes, every time** | The model produces a real answer right now | External monitoring, readiness gates |
| `GET /version` | no | *Which* weights are answering | Provenance stamped onto extracted rows |

The important distinction is between the first two. `/health` is *liveness*: it
reports process state from memory and never touches the GPU. A worker whose CUDA
context has wedged — the classic failure on a shared Jetson where something else
grabbed the memory — will happily keep returning `"status": "ok"` on `/health`
forever, because from the web process's point of view nothing is wrong. It has a
loaded model object; it just cannot use it.

`/health/deep` closes that gap by running an actual forward pass on every call
and returning `503` with `"status": "degraded"` when the pass fails or times
out. Because failure is expressed as a status code rather than a field in a
200 body, `curl -f http://.../health/deep` is a complete check with no JSON
parsing.

The subtle part is the third property, and it is the reason `/health/deep` is
usable at all on a box that only runs one inference at a time. The probe
**deliberately bypasses the inference semaphore**. If it queued like a normal
request, then during any sustained batch job the probe would block, time out,
and report `degraded` — and your monitor would page you for a service that was
merely *busy*. By skipping the queue, the probe distinguishes "wedged" from
"busy", which are two conditions with completely different responses: one needs
a restart, the other needs you to leave it alone.

Read `saturated` and `inflight` on the body if you want the busy signal; read
the status code for the wedged signal. They are separate channels on purpose.

In [2]:
health = get("/health")
show(health)

{
  "status": "ok",
  "model_id": "fastino/gliner2.5-base-v1",
  "model_path": "./models/fastino/gliner2.5-base-v1",
  "loaded": true,
  "architecture": "boundary",
  "model_class": "BoundaryExtractor",
  "device": "cuda",
  "gpu": "Orin",
  "max_concurrent_inferences": 1,
  "inflight": 1,
  "saturated": true
}


Fields worth knowing on that body:

- **`loaded`** — is the model resident in memory? With `MODEL_PRELOAD=0` this
  stays `false` until the first inference request pays the load cost. If you see
  `false` here and then a suspiciously slow first extraction, that is why.
- **`device`** — `"cuda"` on a healthy Jetson. `"cpu"` is not a mode you chose;
  it means the GPU was not visible to the container and every request from now
  on will be an order of magnitude slower. Treat it as a fault.
- **`architecture`** — `"span"` or `"boundary"`. See below; this one decides
  which routes exist for you.
- **`model_class`** — the concrete loaded class, `BoundaryExtractor` here. Do
  **not** assert on this field. `AutoExtractor` is the dispatcher rather than
  the loaded class, so the value varies with how the model was constructed.
  Assert on `architecture` instead, which is the stable contract.
- **`inflight`** / **`saturated`** / **`max_concurrent_inferences`** — how many
  requests currently hold an inference slot, whether the next one will queue,
  and how many slots exist. On this deployment there is exactly one slot.

### Architecture decides which routes you have

GLiNER2 ships in two shapes, and this is not cosmetic.

A **span** model scores candidate spans of text independently against your
labels. That is enough for entity extraction, classification and structured
field filling, because each of those only needs to answer "does this piece of
text belong to this label?".

A **boundary** model — the GLiNER2.5 family — instead predicts where entity
boundaries start and end as a first-class step, producing a set of typed,
delimited mentions rather than a bag of independently scored spans. That extra
structure is what makes *relations* possible: to say "Satya Nadella `works_for`
Microsoft" the model needs two identified, addressable arguments to draw an edge
between. A span model has scored fragments, not arguments, so there is nothing
to connect.

The same structure is what makes efficient true batching possible on the four
`*_batch` routes.

Five of the twelve routes therefore require a boundary checkpoint:
`/extract_relations` and all four `*_batch` routes, plus `schema_config.relations`
inside `/extract_multitask`. On a span model those return **501 Not Implemented**
and name the architecture actually loaded, rather than failing obscurely
somewhere in the model code.

The default `MODEL_ID` (`fastino/gliner2.5-base-v1`) is a boundary checkpoint,
so as deployed everything works. But the check below is worth keeping in any
client that touches those routes — it is the difference between a clear startup
error and a confusing 501 in production at 3am.

In [3]:
ARCH = health["architecture"]
IS_BOUNDARY = ARCH == "boundary"

print("architecture                 :", ARCH)
print("boundary-only routes available:", IS_BOUNDARY)

if health.get("device") != "cuda":
    print("\nWARNING: not running on GPU. On a Jetson this is a fault, not a mode.")

if not IS_BOUNDARY:
    print("\nNOTE: notebooks 05 and 06 need a boundary checkpoint and will skip.")

architecture                 : boundary
boundary-only routes available: True


### `/version` is provenance, not a version string

`model_revision` is a stable 16-hex-character fingerprint of the weights on
disk. It is the single most useful field in this whole notebook for anyone
storing extraction output.

Here is the concrete problem it solves. Span boundaries are the *model's*
choice, and they shift between checkpoints — you will see a worked example of
exactly that in notebook 02. Suppose you extract a million rows, upgrade
`MODEL_ID` six months later, and extract a million more into the same table.
Without a revision stamp those two populations are indistinguishable, and any
analysis that spans the boundary is quietly comparing two different models'
opinions. With it, you can partition, re-run, or exclude.

Stamp `model_revision` onto every row you persist. It costs one column.

In [4]:
show(get("/version"))

{
  "gliner2": "2.0.0",
  "model_id": "fastino/gliner2.5-base-v1",
  "model_revision": "4a3138e2432c24b4",
  "architecture": "boundary",
  "model_class": "BoundaryExtractor",
  "torch": "2.8.0"
}


In [5]:
# /health/deep always probes. Read the STATUS CODE, not just the body:
# a failed probe is a 503, which is what makes `curl -f` a complete check.
r = session.get(f"{BASE_URL}/health/deep", timeout=TIMEOUT)
deep = r.json()

print("HTTP", r.status_code, "| status:", deep["status"])
print("probe:", json.dumps(deep.get("probe"), indent=2))

HTTP 200 | status: ok
probe: {
  "ok": true,
  "latency_ms": 594.5,
  "result": {
    "entities": {
      "company": [
        "Apple"
      ]
    }
  }
}


The `probe` object carries the latency of the real forward pass and the result
it produced. That latency is the number to trend in a monitor: it is
end-to-end model health in one figure, and because the probe bypasses the
semaphore it is not polluted by queueing.

For context on why the bypass matters, one measurement taken on `jarvita-agx`:
while a 48-document batch held the only inference slot, `/health/deep` returned
`200` in 1.56 s while a normal request queued for 4.08 s. That is a single
measurement on a shared box — it demonstrates that the bypass works, and it is
not a latency SLO.

## Your first extraction

Everything above was reconnaissance. Here is the actual point of the service.

`POST /extract_entities` takes two required fields: `text` and `labels`. Notice
what is *not* in the request — no model name, no schema id, no prior
registration step. The labels are the schema, and they exist only for the
duration of this one request.

In [6]:
smoke = post("/extract_entities", {
    "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
    "labels": ["medication", "dosage", "symptom", "time"],
})
show(smoke)

{
  "entities": {
    "medication": [
      "ibuprofen"
    ],
    "dosage": [
      "400mg"
    ],
    "symptom": [
      "severe headache"
    ],
    "time": [
      "2 PM"
    ]
  }
}


Three things to read out of that response, because each one is a rule that holds
for the rest of the path.

**1. The response keys are exactly the labels you sent.** Not a normalized
version, not a subset — the same strings, in a dictionary under `entities`. A
label that matched nothing is still present, mapped to an empty list. That is
deliberate: it means your parsing code can index by label unconditionally
instead of writing `.get(label, [])` everywhere, and it means "no match" is
distinguishable from "I forgot to ask".

**2. Values are bare strings by default.** `"ibuprofen"`, not
`{"text": "ibuprofen", ...}`. This is the default response shape, and notebook
04 covers the two flags that change it. Whichever shape you choose, choose it
once per consumer — a client that flips those flags mid-stream writes two
incompatible shapes into the same dataset.

**3. The model chose the span boundaries, and they are not yours.** Look at the
`symptom` value: it is `"severe headache"`, not `"headache"`. That is a
defensible reading — the severity is part of the symptom — but it is the
*model's* reading, and it will not necessarily match a controlled vocabulary you
hold downstream. Any pipeline that does exact string matching against a term
list needs to account for this, and needs to re-check it whenever `MODEL_ID`
changes. This is the single most common way a GLiNER pipeline breaks silently.

## Try this yourself

Send the same clinical sentence with a label that has nothing to match —
something like `"surgeon"` — alongside the four you already used. Before you run
it, predict: does the response omit the key, return `null`, or return an empty
list?

Then try the opposite: send a label set with a *near-duplicate*, such as
`["medication", "drug"]`, and see what the same text gives you.

In [7]:
show(post("/extract_entities", {
    "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
    "labels": ["medication", "dosage", "symptom", "time", "surgeon"],
}))

print("--- near-duplicate labels ---")
show(post("/extract_entities", {
    "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
    "labels": ["medication", "drug"],
}))

{
  "entities": {
    "medication": [
      "ibuprofen"
    ],
    "dosage": [
      "400mg"
    ],
    "symptom": [
      "severe headache"
    ],
    "time": [
      "2 PM"
    ],
    "surgeon": []
  }
}
--- near-duplicate labels ---


{
  "entities": {
    "medication": [
      "ibuprofen"
    ],
    "drug": [
      "ibuprofen"
    ]
  }
}


**Discussion.** The unmatched label comes back as a key with an empty list, per
rule 1 above — the response shape is determined by your request, not by what was
found.

The near-duplicate case is more interesting, and it is your first encounter with
how these models actually behave. `"medication"` and `"drug"` are not mutually
exclusive categories that the model arbitrates between; they are two independent
questions asked of the same text, each scored on its own. A span can satisfy
both and be returned under both. There is no softmax across your labels forcing
a single winner.

The practical consequence: **your labels are not a partition.** If your
downstream code assumes each extracted span belongs to exactly one category, you
have to enforce that yourself — either by keeping labels genuinely disjoint, or
by deduplicating on the span text after the fact. The service will not do it for
you, because it does not know which of your labels you consider mutually
exclusive.

## What you learned

- `GLINER_BASE_URL` configures every notebook in this path; a client timeout
  above the server's own lets the server report failures as status codes.
- `/health` is liveness and lies about a wedged worker. `/health/deep` runs a
  real forward pass, bypasses the inference semaphore so busy is
  distinguishable from wedged, and reports failure as `503`. That is the one to
  monitor.
- `/version.model_revision` is a weight fingerprint. Stamp it onto every row
  you persist, because span boundaries move between checkpoints.
- `architecture` is `span` or `boundary`. Boundary models identify delimited
  arguments, which is what makes relation extraction and the batch routes
  possible; five of the twelve routes return `501` without one. Assert on
  `architecture`, never on `model_class`.
- Zero-shot means the labels travel with the request. Response keys mirror your
  labels exactly, unmatched labels return empty lists, values are bare strings
  by default, and your label set is *not* a partition.

## Next

**[02 — Extraction and classification](02-extraction-and-classification.ipynb)** —
entity extraction in depth, steering the model with label descriptions, and the
classification route (including the one payload shape it refuses).